# How close is every HBCU to R1?
### Benchmarking research, doctoral output and student outcomes across the 102 HBCUs

*Alton J. Henley*

In February 2025, Howard University became the only HBCU to hold the R1 Carnegie designation, the highest research classification for U.S. universities. Under the 2025 rules, an R1 university spends at least **$50 million a year on research and development** and awards at least **70 research doctorates a year**.

This notebook rebuilds those two measures from public federal data for every HBCU and asks three questions:

1. Which HBCUs meet the R1 thresholds on the most recent data, and which are closest?
2. How fast is HBCU research growing, and who pays for it?
3. How do the leading research HBCUs compare on student outcomes?

**Data.** NSF Higher Education Research and Development (HERD) Survey, FY2022–FY2024, and U.S. Department of Education IPEDS (completions, institutional characteristics, retention and graduation rates). Every figure here can be traced to a public file.

## 1. Setup

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# Interactive charts in Jupyter. Set PLOTLY_RENDERER="notebook_connected+png" to also save
# static images (needs the kaleido package), so charts show up when the notebook is viewed on GitHub.
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook_connected")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

R1_RD = 50_000      # $50M, in thousands of dollars (HERD's unit)
R1_DOC = 70         # research doctorates per year

# Chart palette: one blue for the focus institution, orange for R1/benchmark references,
# gray for other HBCUs.
FOCUS, REFERENCE, PEER = "#2a78d6", "#eb6834", "#a9b0ba"
INK, INK_2, GRID = "#1b2433", "#4f5b6b", "#e6eaef"

def style(fig, height=460, **kw):
    fig.update_layout(height=height, template="none", plot_bgcolor="white", paper_bgcolor="white",
                      font=dict(family="Public Sans, Helvetica, Arial, sans-serif", color=INK, size=13),
                      margin=dict(l=60, r=30, t=60, b=50),
                      legend=dict(orientation="h", y=1.08, x=0, font_color=INK_2), **kw)
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID, tickfont_color=INK_2)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID, tickfont_color=INK_2, automargin=True)
    return fig

print("Raw data folder:", RAW)
print(sorted(p.name for p in RAW.glob("*.csv")))

Raw data folder: /Volumes/Nevvie/Library/Dropbox/hbcudashboard/hbcu-r1-dashboard/data/raw
['C2023_a.csv', 'C2023_a_RV.csv', 'C2024_a.csv', 'HD2024.csv', 'c2022_a.csv', 'c2022_a_rv.csv', 'drvgr2023.csv', 'ef2023d.csv', 'herd2022.csv', 'herd2023.csv', 'herd2024.csv']


## 2. Load and clean the data

### 2.1 Which institutions? (IPEDS directory)

The IPEDS institutional characteristics file flags every HBCU (`HBCU = 1`) and carries the 2025 Carnegie research designation (`CARNEGIERSCH`: 1 = R1, 2 = R2, 3 = Research College/University). I keep all HBCUs, plus the R1s for context.

In [2]:
def read_ipeds(path, **kw):
    df = pd.read_csv(path, encoding="latin-1", low_memory=False, **kw)
    df.columns = [c.replace("ï»¿", "").strip().upper() for c in df.columns]
    return df

hd = read_ipeds(RAW / "HD2024.csv")
designation = {1: "R1", 2: "R2", 3: "Research College/University"}

inst = (hd[(hd.HBCU == 1) | (hd.CARNEGIERSCH == 1)]
        .assign(hbcu=lambda d: d.HBCU.eq(1),
                designation=lambda d: d.CARNEGIERSCH.map(designation).fillna("Not designated"),
                control=lambda d: d.CONTROL.map({1: "Public", 2: "Private nonprofit"}))
        [["UNITID", "INSTNM", "CITY", "STABBR", "hbcu", "designation", "control"]]
        .rename(columns={"UNITID": "unitid", "INSTNM": "name", "CITY": "city", "STABBR": "state"}))

print(f"{inst.hbcu.sum()} HBCUs and {(inst.designation == 'R1').sum()} R1 universities")
inst[inst.hbcu].designation.value_counts()

102 HBCUs and 187 R1 universities


designation
Not designated                 62
Research College/University    26
R2                             13
R1                              1
Name: count, dtype: int64

### 2.2 Research spending (NSF HERD)

HERD's public-use files are in "long" format: one row per institution, question and answer cell. Total R&D spending is question 1, row g, and rows a–f split it by source of funds. Amounts are in **thousands of dollars**.

In [3]:
SOURCES = {"01.a": "federal", "01.b": "state_local", "01.c": "business",
           "01.d": "nonprofit", "01.e": "institution", "01.f": "other", "01.g": "total"}

sample = pd.read_csv(RAW / "herd2024.csv", nrows=200_000, low_memory=False)
sample[(sample.ipeds_unitid == 131520) & sample.questionnaire_no.isin(SOURCES)][
    ["inst_name_long", "year", "questionnaire_no", "row", "data"]]

,inst_name_long,year,questionnaire_no,row,data
41636,Howard University,2024,01.a,Federal government,"55,361.0"
41637,Howard University,2024,01.b,State and local government,"5,844.0"
41638,Howard University,2024,01.c,Business,"6,449.0"
41639,Howard University,2024,01.d,Nonprofit organizations,"9,347.0"
41640,Howard University,2024,01.e,Institution funds,"23,909.0"
41641,Howard University,2024,01.f,All other sources,905.0
41642,Howard University,2024,01.g,Total,"101,815.0"


In [4]:
frames = []
for year in (2022, 2023, 2024):
    h = pd.read_csv(RAW / f"herd{year}.csv", low_memory=False,
                    usecols=["ipeds_unitid", "questionnaire_no", "data"])
    h = h[h.questionnaire_no.isin(SOURCES) & h.ipeds_unitid.isin(inst.unitid)]
    wide = h.pivot_table(index="ipeds_unitid", columns="questionnaire_no", values="data", aggfunc="sum")
    frames.append(wide.rename(columns=SOURCES).assign(fy=year).reset_index())

rd = (pd.concat(frames).rename(columns={"ipeds_unitid": "unitid"})
        [["unitid", "fy", *SOURCES.values()]])
rd.head()

questionnaire_no,unitid,fy,federal,state_local,business,nonprofit,institution,other,total
0,"100,654.0",2022,"9,221.0",4.0,386.0,NaN,NaN,NaN,"9,611.0"
1,"100,663.0",2022,"421,223.0","8,816.0","73,595.0","35,196.0","172,809.0","1,841.0","713,480.0"
2,"100,724.0",2022,"2,930.0",NaN,216.0,1.0,NaN,31.0,"3,178.0"
3,"100,751.0",2022,"67,986.0","17,040.0","2,701.0","1,631.0","73,435.0",411.0,"163,204.0"
4,"100,858.0",2022,"104,883.0","33,555.0","8,338.0","1,912.0","143,876.0","11,875.0","304,439.0"


### 2.3 Research doctorates (IPEDS completions)

A research doctorate is IPEDS award level 17 ("Doctor's degree – research/scholarship"). I take the all-fields total (CIP code 99) for each institution's first major. Where IPEDS has published a revised file (`_RV`), I use it. That matters here: Howard's FY2023 count is 102 in the original file and 96 in the revised one.

Fiscal years line up one to one: HERD FY2024 and IPEDS completions 2024 both cover July 2023 to June 2024 for most institutions.

In [5]:
def completions_file(year):
    for name in (f"C{year}_A_RV.csv", f"c{year}_a_rv.csv", f"C{year}_a_RV.csv", f"C{year}_A.csv", f"C{year}_a.csv", f"c{year}_a.csv"):
        if (RAW / name).exists():
            return RAW / name
    raise FileNotFoundError(f"No completions file for {year}")

docs = []
for year in (2022, 2023, 2024):
    path = completions_file(year)
    c = read_ipeds(path, usecols=lambda x: x.replace("ï»¿", "").strip().upper()
                   in {"UNITID", "CIPCODE", "MAJORNUM", "AWLEVEL", "CTOTALT"})
    c = c[(c.AWLEVEL == 17) & (c.MAJORNUM == 1) & (pd.to_numeric(c.CIPCODE, errors="coerce") == 99)]
    docs.append(pd.DataFrame({"unitid": c.UNITID, "fy": year, "doctorates": c.CTOTALT}))
    print(f"FY{year}: {path.name}")

# No level-17 row means zero research doctorates that year.
grid = pd.MultiIndex.from_product([inst.unitid, [2022, 2023, 2024]], names=["unitid", "fy"]).to_frame(index=False)
doc = grid.merge(pd.concat(docs), how="left").fillna({"doctorates": 0})

FY2022: C2022_A_RV.csv
FY2023: C2023_A_RV.csv
FY2024: C2024_A.csv


### 2.4 Student outcomes (IPEDS)

In [6]:
ef = read_ipeds(RAW / "ef2023d.csv")
gr = read_ipeds(RAW / "drvgr2023.csv")
num = lambda s: pd.to_numeric(s.astype(str).str.strip().replace(".", None), errors="coerce")

outcomes = (pd.DataFrame({"unitid": ef.UNITID, "retention": num(ef.RET_PCF)})
            .merge(pd.DataFrame({"unitid": gr.UNITID, "grad_6yr": num(gr.GBA6RTT),
                                 "grad_6yr_pell": num(gr.PGBA6RT)}), how="outer"))
print("Retention: first-time, full-time students, fall 2023.  Graduation: bachelor's within 6 years, 2017 entering cohort.")

Retention: first-time, full-time students, fall 2023.  Graduation: bachelor's within 6 years, 2017 entering cohort.


### 2.5 One row per institution

Following the 2025 rules, each institution is measured on its **3-year average** for FY2022–FY2024.

In [7]:
rd3 = rd.groupby("unitid").agg(rd_3yr=("total", "mean"))
doc3 = doc.groupby("unitid").agg(doc_3yr=("doctorates", "mean"))
latest = rd[rd.fy == 2024].set_index("unitid")
first = rd[rd.fy == 2022].set_index("unitid")

df = (inst.set_index("unitid")
      .join(rd3).join(doc3)
      .assign(rd_2024=latest.total,
              rd_growth=latest.total / first.total - 1,
              federal_share=latest.federal / latest.total,
              doc_2024=doc[doc.fy == 2024].set_index("unitid").doctorates)
      .join(outcomes.set_index("unitid"))
      .assign(rd_3yr_m=lambda d: d.rd_3yr / 1000,
              pct_rd=lambda d: d.rd_3yr / R1_RD,
              pct_doc=lambda d: d.doc_3yr / R1_DOC,
              meets_rd=lambda d: d.rd_3yr >= R1_RD,
              meets_doc=lambda d: d.doc_3yr >= R1_DOC)
      .assign(readiness=lambda d: d[["pct_rd", "pct_doc"]].min(axis=1))
      .reset_index())

hbcu = df[df.hbcu].copy()
r1 = df[df.designation == "R1"].copy()
print(f"{hbcu.rd_3yr.notna().sum()} of {len(hbcu)} HBCUs appear in HERD's full survey (R&D of $1M or more) "
      "in at least one of the three years.")

50 of 102 HBCUs appear in HERD's full survey (R&D of $1M or more) in at least one of the three years.


## 3. Validation: does this match the official numbers?

When ACE announced Howard's R1 designation, it reported that in FY2023, the evaluation year, Howard spent just under **$85 million** on research and awarded **96 research doctorates**. If the pipeline is right, it should reproduce both.

In [8]:
howard = 131520
check = pd.DataFrame({
    "R&D ($M)": rd[rd.unitid == howard].set_index("fy").total / 1000,
    "Research doctorates": doc[doc.unitid == howard].set_index("fy").doctorates,
})
display(check)

assert abs(check.loc[2023, "R&D ($M)"] - 84.8) < 0.5, "R&D does not match ACE's reported figure"
assert check.loc[2023, "Research doctorates"] == 96, "Doctorates do not match ACE's reported figure"
print("✓ Reproduces ACE's published FY2023 figures for Howard: $84.8M in R&D and 96 research doctorates.")

,R&D ($M),Research doctorates
fy,,
2022,67.6,86.0
2023,84.8,96.0
2024,101.8,102.0


✓ Reproduces ACE's published FY2023 figures for Howard: $84.8M in R&D and 96 research doctorates.


## 4. Who meets the R1 thresholds?

Each dot below is an HBCU, placed by its 3-year average research spending and research doctorates. Schools above and to the right of both dashed lines meet the R1 thresholds.

In [9]:
plot = hbcu.dropna(subset=["rd_3yr"])
lead = plot.nlargest(5, "readiness")

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot.doc_3yr, y=plot.rd_3yr_m, mode="markers", name="HBCUs",
                         text=plot.name, marker=dict(size=9, color=PEER, line=dict(color="white", width=1.5)),
                         hovertemplate="<b>%{text}</b><br>R&D: $%{y:.1f}M<br>Doctorates: %{x:.1f}<extra></extra>"))
h = plot[plot.unitid == howard]
fig.add_trace(go.Scatter(x=h.doc_3yr, y=h.rd_3yr_m, mode="markers", name="Howard University",
                         marker=dict(size=14, color=FOCUS, line=dict(color="white", width=2)),
                         hovertemplate="<b>Howard University</b><br>R&D: $%{y:.1f}M<br>Doctorates: %{x:.1f}<extra></extra>"))
for _, r in lead.iterrows():
    fig.add_annotation(x=r.doc_3yr, y=r.rd_3yr_m, text=r["name"], showarrow=False,
                       xanchor="right", xshift=-10, font=dict(size=11, color=INK_2))
fig.add_hline(y=50, line=dict(color=INK_2, dash="dash", width=1.5),
              annotation_text="$50M R&D threshold", annotation_position="top left")
fig.add_vline(x=70, line=dict(color=INK_2, dash="dash", width=1.5),
              annotation_text="70 doctorates", annotation_position="bottom right")
style(fig, height=540, title="HBCUs against the R1 thresholds (3-year averages, FY2022–FY2024)")
fig.update_xaxes(title="Research doctorates per year")
fig.update_yaxes(title="R&D expenditures, $ millions")
fig.show()

In [10]:
near = (plot.sort_values("readiness", ascending=False).head(10)
        .assign(rd_gap_m=lambda d: (50 - d.rd_3yr_m).clip(lower=0),
                doc_gap=lambda d: (70 - d.doc_3yr).clip(lower=0))
        [["name", "designation", "rd_3yr_m", "rd_gap_m", "doc_3yr", "doc_gap", "readiness"]]
        .rename(columns={"name": "Institution", "designation": "2025 designation",
                         "rd_3yr_m": "R&D 3-yr avg ($M)", "rd_gap_m": "R&D gap ($M)",
                         "doc_3yr": "Doctorates 3-yr avg", "doc_gap": "Doctorate gap",
                         "readiness": "Readiness"}))
near.style.format({"R&D 3-yr avg ($M)": "{:,.1f}", "R&D gap ($M)": "{:,.1f}", "Doctorates 3-yr avg": "{:,.1f}",
                   "Doctorate gap": "{:,.1f}",
                   # don't round a near-miss up to 100%
                   "Readiness": lambda v: "100%" if v >= 1 else (f"{v:.1%}" if v >= 0.995 else f"{v:.0%}")}).hide(axis="index")

Institution,2025 designation,R&D 3-yr avg ($M),R&D gap ($M),Doctorates 3-yr avg,Doctorate gap,Readiness
Howard University,R1,84.7,0.0,94.7,0.0,100%
North Carolina A & T State University,R2,64.0,0.0,69.7,0.3,99.5%
Morgan State University,R2,43.0,7.0,63.7,6.3,86%
Tennessee State University,R2,27.1,22.9,44.7,25.3,54%
Florida Agricultural and Mechanical University,R2,61.5,0.0,35.3,34.7,50%
Jackson State University,R2,20.6,29.4,51.0,19.0,41%
Delaware State University,R2,35.4,14.6,28.0,42.0,40%
Prairie View A & M University,R2,24.6,25.4,27.3,42.7,39%
Texas Southern University,R2,15.8,34.2,31.0,39.0,32%
Virginia State University,R2,13.0,37.0,18.0,52.0,26%


In [11]:
both = plot[plot.meets_rd & plot.meets_doc]
nca = plot[plot.name.str.startswith("North Carolina A & T")].iloc[0]
print(f"HBCUs meeting both thresholds: {', '.join(both.name)}")
print(f"North Carolina A&T: ${nca.rd_3yr_m:.1f}M in R&D (meets) and {nca.doc_3yr:.1f} doctorates a year, "
      f"{70 - nca.doc_3yr:.1f} short of 70.")

HBCUs meeting both thresholds: Howard University
North Carolina A&T: $64.0M in R&D (meets) and 69.7 doctorates a year, 0.3 short of 70.


**Takeaway.** Howard is the only HBCU above both thresholds on FY2022–FY2024 data. North Carolina A&T is closest: it clears the R&D bar and falls short of 70 research doctorates by a fraction of a degree a year, consistent with reports that it narrowly missed R1 in 2025. These positions are computed on newer data than the 2025 designations, so they show where institutions stand today, not their official status.

## 5. How fast is HBCU research growing?

In [12]:
hb_rd = rd[rd.unitid.isin(hbcu.unitid)].groupby("fy").total.sum() / 1e6
r1_rd = rd[rd.unitid.isin(r1.unitid)].groupby("fy").total.sum() / 1e6
growth = pd.DataFrame({"All HBCUs ($B)": hb_rd, "All R1 universities ($B)": r1_rd})
growth.index = [f"FY{y}" for y in growth.index]
display(growth.style.format("{:,.2f}"))
change = growth.iloc[-1] / growth.iloc[0] - 1
print(f"Growth FY2022→FY2024: HBCUs {change.iloc[0]:+.1%}, R1 universities {change.iloc[1]:+.1%}")

,All HBCUs ($B),All R1 universities ($B)
FY2022,0.63,74.55
FY2023,0.76,82.23
FY2024,0.88,89.22


Growth FY2022→FY2024: HBCUs +40.3%, R1 universities +19.7%


In [13]:
top = hbcu.nlargest(10, "rd_2024")[["name", "rd_2024", "rd_growth", "federal_share"]].sort_values("rd_growth", ascending=False)
fig = go.Figure(go.Bar(
    x=top.rd_growth[::-1], y=top.name[::-1], orientation="h",
    marker=dict(color=[FOCUS if n == "Howard University" else PEER for n in top.name[::-1]], cornerradius=4),
    text=[f"{v:+.0%}" for v in top.rd_growth[::-1]], textposition="outside", cliponaxis=False,
    hovertemplate="<b>%{y}</b><br>Growth: %{x:.0%}<extra></extra>"))
style(fig, height=430, title="R&D growth, FY2022 to FY2024: the 10 largest HBCU research programs", showlegend=False)
fig.update_xaxes(tickformat=".0%", title="Change in total R&D spending")
fig.show()
med = hbcu.rd_growth.median()
fastest = top.nlargest(3, "rd_growth")
print(f"Median HBCU growth: {med:+.0%}. All 10 of the largest programs grew faster than that.")
print("Fastest among the 10 largest: " + ", ".join(f"{r['name']} ({r.rd_growth:+.0%})" for _, r in fastest.iterrows()))

Median HBCU growth: +26%. All 10 of the largest programs grew faster than that.
Fastest among the 10 largest: Tennessee State University (+94%), Morgan State University (+88%), North Carolina A & T State University (+71%)


**Takeaway.** HBCU research spending is growing faster than research spending at R1 universities overall. Every one of the 10 largest HBCU research programs grew faster than the median HBCU, so growth is concentrated at the top. That suggests more HBCUs could approach the R1 thresholds in the next review cycle, expected in 2028.

## 6. Who pays for HBCU research?

In [14]:
fs = hbcu.dropna(subset=["federal_share"])
fig = go.Figure()
fig.add_trace(go.Histogram(x=fs.federal_share, nbinsx=20, marker=dict(color=PEER, line=dict(color="white", width=1)),
                           name="HBCUs", hovertemplate="%{x}: %{y} HBCUs<extra></extra>"))
hv = fs[fs.unitid == howard].federal_share.iloc[0]
fig.add_vline(x=hv, line=dict(color=FOCUS, width=3), annotation_text=f"Howard: {hv:.0%}",
              annotation_position="top left", annotation_font_color=FOCUS)
fig.add_vline(x=fs.federal_share.median(), line=dict(color=REFERENCE, dash="dot", width=2),
              annotation_text=f"HBCU median: {fs.federal_share.median():.0%}", annotation_position="top right")
style(fig, height=400, title="Federal share of R&D spending, FY2024", showlegend=False, bargap=0.05)
fig.update_xaxes(tickformat=".0%", title="Share of R&D funded by the federal government")
fig.update_yaxes(title="Number of HBCUs")
fig.show()
print(f"Howard relies less on federal funding than {(fs.federal_share > hv).sum()} of {len(fs)} HBCUs in HERD.")

Howard relies less on federal funding than 40 of 47 HBCUs in HERD.


**Takeaway.** Most HBCU research depends heavily on federal agencies. Howard's mix is more diversified: about half federal, with substantial institutional, nonprofit and business funding. That's an advantage when federal research budgets are uncertain.

## 7. Research scale and student outcomes

In [15]:
o = hbcu.dropna(subset=["grad_6yr"]).assign(rd_plot=lambda d: d.rd_3yr_m.fillna(0.5))
fig = go.Figure()
fig.add_trace(go.Scatter(x=o.rd_plot, y=o.grad_6yr, mode="markers", text=o.name, name="HBCUs",
                         marker=dict(size=9, color=PEER, line=dict(color="white", width=1.5)),
                         hovertemplate="<b>%{text}</b><br>R&D: $%{x:.1f}M<br>6-yr grad rate: %{y}%<extra></extra>"))
oh = o[o.unitid == howard]
fig.add_trace(go.Scatter(x=oh.rd_plot, y=oh.grad_6yr, mode="markers+text", text=["Howard"], name="Howard University",
                         textposition="top left", marker=dict(size=14, color=FOCUS, line=dict(color="white", width=2))))
style(fig, height=460, title="R&D scale vs. 6-year graduation rate")
fig.update_xaxes(type="log", title="R&D expenditures, $ millions (3-yr average, log scale; under $1M plotted at $0.5M)")
fig.update_yaxes(title="6-year bachelor's graduation rate (%)")
fig.show()
corr = np.corrcoef(np.log(o.rd_plot), o.grad_6yr)[0, 1]
print(f"Correlation between log R&D and graduation rate: {corr:.2f}")

Correlation between log R&D and graduation rate: 0.32


**Takeaway.** Research scale and graduation rates are positively related across HBCUs, but the spread is wide. Several smaller institutions graduate students at high rates, and research intensity alone doesn't explain student success. Howard ranks near the top on both.

## 8. Scorecard for any HBCU

Change the name in the next cell to see any HBCU's position on every measure.

In [16]:
pct0 = lambda v: f"{v:.0%}"
signed = lambda v: f"{v:+.0%}"
MEASURES = {  # label: (column, scale, formatter)
    "R&D, FY2024": ("rd_2024", 1e-3, lambda v: f"${v:,.1f}M"),
    "R&D growth, FY2022 to FY2024": ("rd_growth", 1, signed),
    "Federal share of R&D": ("federal_share", 1, pct0),
    "Research doctorates, FY2024": ("doc_2024", 1, lambda v: f"{v:,.0f}"),
    "First-year retention": ("retention", 0.01, pct0),
    "6-year graduation rate": ("grad_6yr", 0.01, pct0),
    "6-year graduation rate, Pell recipients": ("grad_6yr_pell", 0.01, pct0),
}

def scorecard(name):
    row = hbcu[hbcu.name == name].iloc[0]
    out = []
    for label, (col, scale, f) in MEASURES.items():
        s, v = hbcu[col].dropna(), row[col]
        out.append({"Measure": label,
                    "Value": "—" if pd.isna(v) else f(v * scale),
                    "HBCU median": f(s.median() * scale),
                    "Rank": "—" if pd.isna(v) else f"{(s > v).sum() + 1} of {len(s)}",
                    "Percentile": np.nan if pd.isna(v) else (s < v).mean()})
    print(f"{name}: {row.city}, {row.state}. 2025 designation: {row.designation}")
    return (pd.DataFrame(out).style.hide(axis="index")
            .format({"Percentile": lambda v: "—" if pd.isna(v) else f"{v:.0%}"})
            .bar(subset=["Percentile"], color="#b7d3f6", vmin=0, vmax=1)
            .set_properties(subset=["Measure"], **{"text-align": "left"}))

scorecard("Howard University")

Howard University: Washington, DC. 2025 designation: R1


Measure,Value,HBCU median,Rank,Percentile
"R&D, FY2024",$101.8M,$12.5M,1 of 47,98%
"R&D growth, FY2022 to FY2024",+51%,+26%,15 of 45,67%
Federal share of R&D,54%,72%,41 of 47,13%
"Research doctorates, FY2024",102,0,1 of 102,99%
First-year retention,90%,64%,2 of 96,96%
6-year graduation rate,69%,32%,4 of 87,95%
"6-year graduation rate, Pell recipients",63%,29%,6 of 87,93%


In [17]:
scorecard("Morgan State University")

Morgan State University: Baltimore, MD. 2025 designation: R2


Measure,Value,HBCU median,Rank,Percentile
"R&D, FY2024",$55.5M,$12.5M,5 of 47,89%
"R&D growth, FY2022 to FY2024",+88%,+26%,4 of 45,91%
Federal share of R&D,67%,72%,28 of 47,40%
"Research doctorates, FY2024",62,0,3 of 102,97%
First-year retention,71%,64%,30 of 96,65%
6-year graduation rate,42%,32%,24 of 87,72%
"6-year graduation rate, Pell recipients",38%,29%,29 of 87,64%


## 9. Summary

- **Howard is the only HBCU that meets both R1 thresholds** on FY2022–FY2024 data, at about 169% of the R&D threshold and 135% of the doctorate threshold.
- **North Carolina A&T is within a fraction of a doctorate a year of R1.** It already clears the R&D bar. Morgan State is next.
- **HBCU research is growing faster than R1 research overall,** and the fastest growth is concentrated in the leading programs.
- **Howard's research funding is more diversified than most HBCUs',** with about half from federal sources against an HBCU median near 72%.
- **Research scale and graduation rates move together, but loosely.** Student success at HBCUs isn't simply a function of research intensity.

### Limitations
- Positions are computed from public files and on newer data than the 2025 designations. They show where institutions stand today, not their official status.
- HBCUs with under $1M in R&D answer HERD's short form, which isn't included here.
- Dollar amounts are in current, not inflation-adjusted, dollars.

### Sources
- National Center for Science and Engineering Statistics, Higher Education Research and Development Survey, public-use files, FY2022–FY2024.
- U.S. Department of Education, NCES, IPEDS: HD2024, C2022–C2024 completions (revised files where available), EF2023D and DRVGR2023.

The companion Streamlit dashboard (`app.py`) presents the same data interactively.